<a href="https://colab.research.google.com/github/san-258/Pullback-to-21-ema-scan-for-D-1Hr-15min/blob/Scanner/Pullback_to_21_ema%2C_scan_for_D_1Hr_15min.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ** All work done by Cloude, this is scanner of on 70 of NASDAQ stocks

**

# These 3 scanning criteria listed below scanning for hammer candle near 21 ema

1st one is On daily scanner

2nd is on hourly scanner

3rd one is on 15 min




## I have 4th one down, scanning for candlestick pattern at or around 21 ema, with volume, decription is above it  

The 4th one is powerfull, it scan all stocks, -   if it in trend or not, volume, distance near 21 ema, candle stick pattern, strenght of pattern, and whole lot other information, check results,

# Task
Scan the provided list of tickers Daily for the following setups:

Trend confirmation: 50 EMA > 200 EMA, price above 21 EMA

Setup A: Pullback Buy: Price touches 21 EMA, RSI 30-40, Bullish reversal candle i.e Hammer, Bullish engulfing, Volume 1.5X of 20 ma


In [1]:
pip install yfinance pandas numpy

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

class TechnicalScanner:
    def __init__(self, tickers):
        self.tickers = list(set(tickers))  # Remove duplicates
        self.results = []

    def calculate_ema(self, prices, period):
        """Calculate Exponential Moving Average"""
        return prices.ewm(span=period).mean()

    def calculate_rsi(self, prices, period=14):
        """Calculate RSI"""
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    def is_hammer(self, open_price, high, low, close):
        """Identify Hammer candlestick pattern"""
        body = abs(close - open_price)
        upper_shadow = high - max(open_price, close)
        lower_shadow = min(open_price, close) - low

        # Hammer criteria: small body, long lower shadow, small upper shadow
        return (lower_shadow > 2 * body and
                upper_shadow < body and
                body > 0)

    def is_bullish_engulfing(self, prev_open, prev_close, curr_open, curr_close):
        """Identify Bullish Engulfing pattern"""
        prev_bearish = prev_close < prev_open
        curr_bullish = curr_close > curr_open
        engulfing = curr_open < prev_close and curr_close > prev_open

        return prev_bearish and curr_bullish and engulfing

    def analyze_stock(self, ticker):
        """Analyze individual stock for trading setup"""
        try:
            # Fetch data for the last 6 months to ensure enough data for indicators
            stock = yf.Ticker(ticker)
            hist = stock.history(period="6mo")

            if len(hist) < 200:  # Need at least 200 days for 200 EMA
                return None

            # Calculate indicators
            hist['EMA_21'] = self.calculate_ema(hist['Close'], 21)
            hist['EMA_50'] = self.calculate_ema(hist['Close'], 50)
            hist['EMA_200'] = self.calculate_ema(hist['Close'], 200)
            hist['RSI'] = self.calculate_rsi(hist['Close'])
            hist['Volume_MA_20'] = hist['Volume'].rolling(window=20).mean()

            # Get latest values
            latest = hist.iloc[-1]
            prev = hist.iloc[-2]
            current_price = latest['Close']

            # Check trend confirmation
            trend_confirmed = (latest['EMA_50'] > latest['EMA_200'] and
                             current_price > latest['EMA_21'])

            # Check if price is touching/near 21 EMA (within 2%)
            price_near_ema21 = abs(current_price - latest['EMA_21']) / latest['EMA_21'] <= 0.02

            # Check RSI range
            rsi_in_range = 30 <= latest['RSI'] <= 40

            # Check volume
            volume_above_avg = latest['Volume'] >= (1.5 * latest['Volume_MA_20'])

            # Check for bullish reversal patterns
            hammer = self.is_hammer(latest['Open'], latest['High'], latest['Low'], latest['Close'])
            bullish_engulfing = self.is_bullish_engulfing(prev['Open'], prev['Close'],
                                                         latest['Open'], latest['Close'])
            reversal_candle = hammer or bullish_engulfing

            # Setup A criteria
            setup_a = (price_near_ema21 and rsi_in_range and
                      reversal_candle and volume_above_avg)

            return {
                'ticker': ticker,
                'current_price': round(current_price, 2),
                'ema_21': round(latest['EMA_21'], 2),
                'ema_50': round(latest['EMA_50'], 2),
                'ema_200': round(latest['EMA_200'], 2),
                'rsi': round(latest['RSI'], 2),
                'volume': int(latest['Volume']),
                'volume_ma_20': int(latest['Volume_MA_20']),
                'trend_confirmed': trend_confirmed,
                'price_near_ema21': price_near_ema21,
                'rsi_in_range': rsi_in_range,
                'volume_above_avg': volume_above_avg,
                'hammer': hammer,
                'bullish_engulfing': bullish_engulfing,
                'reversal_candle': reversal_candle,
                'setup_a_qualified': setup_a,
                'distance_from_ema21_pct': round(((current_price - latest['EMA_21']) / latest['EMA_21']) * 100, 2)
            }

        except Exception as e:
            print(f"Error analyzing {ticker}: {str(e)}")
            return None

    def scan_all_stocks(self):
        """Scan all stocks in the ticker list"""
        print("Starting technical analysis scan...")
        print(f"Analyzing {len(self.tickers)} unique tickers...")

        for i, ticker in enumerate(self.tickers):
            print(f"Analyzing {ticker} ({i+1}/{len(self.tickers)})")
            result = self.analyze_stock(ticker)
            if result:
                self.results.append(result)

        return self.results

    def generate_report(self):
        """Generate detailed report"""
        if not self.results:
            print("No results to report")
            return

        df = pd.DataFrame(self.results)

        print("\n" + "="*80)
        print("TECHNICAL ANALYSIS SCAN RESULTS")
        print("="*80)

        # Setup A qualified stocks
        setup_a_stocks = df[df['setup_a_qualified'] == True]
        print(f"\n🎯 SETUP A QUALIFIED STOCKS ({len(setup_a_stocks)}):")
        print("-" * 50)

        if len(setup_a_stocks) > 0:
            for _, stock in setup_a_stocks.iterrows():
                print(f"{stock['ticker']}: ${stock['current_price']} | RSI: {stock['rsi']} | "
                      f"EMA21 Distance: {stock['distance_from_ema21_pct']}%")
        else:
            print("No stocks currently meet all Setup A criteria")

        # Trend confirmed stocks
        trend_confirmed = df[df['trend_confirmed'] == True]
        print(f"\n📈 TREND CONFIRMED STOCKS ({len(trend_confirmed)}):")
        print("-" * 50)
        print("(50 EMA > 200 EMA and Price > 21 EMA)")

        for _, stock in trend_confirmed.head(10).iterrows():
            print(f"{stock['ticker']}: ${stock['current_price']} | RSI: {stock['rsi']}")

        if len(trend_confirmed) > 10:
            print(f"... and {len(trend_confirmed) - 10} more")

        # Stocks near pullback zone
        near_pullback = df[(df['rsi'] >= 30) & (df['rsi'] <= 50) &
                          (df['distance_from_ema21_pct'] >= -5) &
                          (df['distance_from_ema21_pct'] <= 5)]

        print(f"\n🔄 STOCKS NEAR PULLBACK ZONE ({len(near_pullback)}):")
        print("-" * 50)
        print("(RSI 30-50 and within 5% of 21 EMA)")

        for _, stock in near_pullback.iterrows():
            print(f"{stock['ticker']}: ${stock['current_price']} | RSI: {stock['rsi']} | "
                  f"EMA21 Distance: {stock['distance_from_ema21_pct']}%")

        # Summary statistics
        print(f"\n📊 SUMMARY STATISTICS:")
        print("-" * 50)
        print(f"Total stocks analyzed: {len(df)}")
        print(f"Trend confirmed: {len(trend_confirmed)} ({len(trend_confirmed)/len(df)*100:.1f}%)")
        print(f"Setup A qualified: {len(setup_a_stocks)} ({len(setup_a_stocks)/len(df)*100:.1f}%)")
        print(f"Average RSI: {df['rsi'].mean():.1f}")
        print(f"RSI range: {df['rsi'].min():.1f} - {df['rsi'].max():.1f}")

        return df

# Main execution
if __name__ == "__main__":
    # Your ticker list (removed duplicates)
    tickers = ["AAPL","MSFT","GOOGL","GOOG","AMZN","TSLA","META","NVDA","PYPL","INTC",
               "CSCO","NFLX","PEP","ADBE","AVGO","TXN","QCOM","INTU","AMGN","MDLZ",
               "SBUX","ISRG","GILD","REGN","VRTX","CSX","ADP","ILMN","EXC","BIIB",
               "LULU","IDXX","CTAS","ROST","MAR","MNST","KLAC","CTSH","MU",
               "WBA","JD","BIDU","EXPE","CHTR","EA","FAST","HUM","INCY","KDP","KHC",
               "LRCX","MCHP","MRVL","MTCH","MSCI","NTES","NXPI","ODFL","ORLY","PAYX",
               "PCAR","PDD","SNPS","TTWO","VRSK","WBD","WDAY","XEL","ZS"]

    # Initialize scanner
    scanner = TechnicalScanner(tickers)

    # Run the scan
    results = scanner.scan_all_stocks()

    # Generate and display report
    df_results = scanner.generate_report()

    # Optional: Save results to CSV
    if len(results) > 0:
        df_results.to_csv('technical_scan_results.csv', index=False)
        print(f"\n💾 Results saved to 'technical_scan_results.csv'")

        # Show detailed Setup A candidates
        setup_a = df_results[df_results['setup_a_qualified'] == True]
        if len(setup_a) > 0:
            print("\n🎯 DETAILED SETUP A ANALYSIS:")
            print("="*80)
            for _, stock in setup_a.iterrows():
                print(f"\n{stock['ticker']} - ${stock['current_price']}")
                print(f"  RSI: {stock['rsi']} (Target: 30-40)")
                print(f"  Distance from 21 EMA: {stock['distance_from_ema21_pct']}%")
                print(f"  Volume vs 20-day avg: {stock['volume']/stock['volume_ma_20']:.1f}x")
                print(f"  Reversal candle: {'✅' if stock['reversal_candle'] else '❌'}")
                print(f"  Trend confirmed: {'✅' if stock['trend_confirmed'] else '❌'}")

Starting technical analysis scan...
Analyzing 69 unique tickers...
Analyzing MU (1/69)
Analyzing MSCI (2/69)
Analyzing TSLA (3/69)
Analyzing JD (4/69)
Analyzing CTAS (5/69)
Analyzing MAR (6/69)
Analyzing EXC (7/69)
Analyzing REGN (8/69)
Analyzing EA (9/69)
Analyzing SBUX (10/69)
Analyzing KDP (11/69)
Analyzing ILMN (12/69)
Analyzing MTCH (13/69)
Analyzing MRVL (14/69)
Analyzing PCAR (15/69)
Analyzing AVGO (16/69)
Analyzing CTSH (17/69)
Analyzing VRTX (18/69)
Analyzing AMZN (19/69)
Analyzing INTC (20/69)
Analyzing KLAC (21/69)
Analyzing NXPI (22/69)
Analyzing CSX (23/69)
Analyzing ZS (24/69)
Analyzing AAPL (25/69)
Analyzing ADP (26/69)
Analyzing HUM (27/69)
Analyzing BIDU (28/69)
Analyzing PAYX (29/69)
Analyzing KHC (30/69)
Analyzing MCHP (31/69)
Analyzing ADBE (32/69)
Analyzing AMGN (33/69)
Analyzing PDD (34/69)
Analyzing MSFT (35/69)
Analyzing INTU (36/69)
Analyzing GOOGL (37/69)
Analyzing NVDA (38/69)
Analyzing MNST (39/69)
Analyzing XEL (40/69)
Analyzing MDLZ (41/69)
Analyzing ROST 

In [3]:
results = scanner.scan_all_stocks()
df_results = scanner.generate_report()

Starting technical analysis scan...
Analyzing 69 unique tickers...
Analyzing MU (1/69)
Analyzing MSCI (2/69)
Analyzing TSLA (3/69)
Analyzing JD (4/69)
Analyzing CTAS (5/69)
Analyzing MAR (6/69)
Analyzing EXC (7/69)
Analyzing REGN (8/69)
Analyzing EA (9/69)
Analyzing SBUX (10/69)
Analyzing KDP (11/69)
Analyzing ILMN (12/69)
Analyzing MTCH (13/69)
Analyzing MRVL (14/69)
Analyzing PCAR (15/69)
Analyzing AVGO (16/69)
Analyzing CTSH (17/69)
Analyzing VRTX (18/69)
Analyzing AMZN (19/69)
Analyzing INTC (20/69)
Analyzing KLAC (21/69)
Analyzing NXPI (22/69)
Analyzing CSX (23/69)
Analyzing ZS (24/69)
Analyzing AAPL (25/69)
Analyzing ADP (26/69)
Analyzing HUM (27/69)
Analyzing BIDU (28/69)
Analyzing PAYX (29/69)
Analyzing KHC (30/69)
Analyzing MCHP (31/69)
Analyzing ADBE (32/69)
Analyzing AMGN (33/69)
Analyzing PDD (34/69)
Analyzing MSFT (35/69)
Analyzing INTU (36/69)
Analyzing GOOGL (37/69)
Analyzing NVDA (38/69)
Analyzing MNST (39/69)
Analyzing XEL (40/69)
Analyzing MDLZ (41/69)
Analyzing ROST 

&&&&&&&&&&&&&&&&&&&&&
*********************
$$$$$$$$$$$$$$$$$$$$

Hourly

$$$$$$$$$$$$$$$$$$$$
********************
&&&&&&&&&&&&&&&&&&&&

In [4]:
# Google Colab Stock Scanner - Hourly Technical Analysis
# Run this cell to install required packages
!pip install yfinance pandas numpy schedule plotly

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import json
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output, HTML
import threading
from google.colab import files
warnings.filterwarnings('ignore')

class ColabTechnicalScanner:
    def __init__(self, tickers):
        self.tickers = list(set(tickers))  # Remove duplicates
        self.results = []
        self.previous_results = {}
        self.alerts = []
        self.scan_history = []
        self.is_running = False

    def calculate_ema(self, prices, period):
        """Calculate Exponential Moving Average"""
        return prices.ewm(span=period).mean()

    def calculate_rsi(self, prices, period=14):
        """Calculate RSI"""
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    def is_hammer(self, open_price, high, low, close):
        """Identify Hammer candlestick pattern"""
        body = abs(close - open_price)
        upper_shadow = high - max(open_price, close)
        lower_shadow = min(open_price, close) - low

        return (lower_shadow > 2 * body and
                upper_shadow < body and
                body > 0)

    def is_bullish_engulfing(self, prev_open, prev_close, curr_open, curr_close):
        """Identify Bullish Engulfing pattern"""
        prev_bearish = prev_close < prev_open
        curr_bullish = curr_close > curr_open
        engulfing = curr_open < prev_close and curr_close > prev_open

        return prev_bearish and curr_bullish and engulfing

    def is_doji(self, open_price, close, high, low):
        """Identify Doji pattern"""
        body = abs(close - open_price)
        total_range = high - low
        return body <= (total_range * 0.1) if total_range > 0 else False

    def get_market_data(self, ticker):
        """Get market data optimized for Colab"""
        try:
            stock = yf.Ticker(ticker)

            # Get intraday data (last 2 days, 5-minute intervals)
            intraday = stock.history(period="2d", interval="5m")

            # Get daily data for EMAs
            daily = stock.history(period="6mo", interval="1d")

            return intraday, daily
        except Exception as e:
            return None, None

    def analyze_stock(self, ticker):
        """Analyze individual stock"""
        try:
            intraday, daily = self.get_market_data(ticker)

            if intraday is None or daily is None or len(daily) < 50:
                return None

            # Calculate daily EMAs
            daily['EMA_21'] = self.calculate_ema(daily['Close'], 21)
            daily['EMA_50'] = self.calculate_ema(daily['Close'], 50)
            daily['EMA_200'] = self.calculate_ema(daily['Close'], 200)

            # Resample intraday to hourly
            hourly = intraday.resample('1H').agg({
                'Open': 'first',
                'High': 'max',
                'Low': 'min',
                'Close': 'last',
                'Volume': 'sum'
            }).dropna()

            if len(hourly) < 20:
                return None

            # Calculate hourly indicators
            hourly['EMA_21'] = self.calculate_ema(hourly['Close'], 21)
            hourly['RSI'] = self.calculate_rsi(hourly['Close'], 14)
            hourly['Volume_MA'] = hourly['Volume'].rolling(window=20).mean()

            # Current values
            current = hourly.iloc[-1]
            prev = hourly.iloc[-2] if len(hourly) > 1 else current
            daily_current = daily.iloc[-1]

            current_price = current['Close']

            # Trend confirmation
            trend_confirmed = (daily_current['EMA_50'] > daily_current['EMA_200'] and
                             current_price > daily_current['EMA_21'])

            # Setup criteria
            price_near_ema21 = abs(current_price - current['EMA_21']) / current['EMA_21'] <= 0.03
            rsi_in_range = 25 <= current['RSI'] <= 45
            volume_above_avg = current['Volume'] >= (1.3 * current['Volume_MA'])

            # Reversal patterns
            hammer = self.is_hammer(current['Open'], current['High'], current['Low'], current['Close'])
            bullish_engulfing = self.is_bullish_engulfing(prev['Open'], prev['Close'],
                                                         current['Open'], current['Close'])
            doji = self.is_doji(current['Open'], current['Close'], current['High'], current['Low'])
            reversal_candle = hammer or bullish_engulfing or (doji and current['RSI'] < 40)

            # Setup A qualification
            setup_a = (price_near_ema21 and rsi_in_range and
                      reversal_candle and volume_above_avg and trend_confirmed)

            # Momentum score
            momentum_score = self.calculate_momentum(current, hourly)

            return {
                'ticker': ticker,
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'price': round(current_price, 2),
                'ema_21': round(current['EMA_21'], 2),
                'daily_ema_50': round(daily_current['EMA_50'], 2),
                'daily_ema_200': round(daily_current['EMA_200'], 2),
                'rsi': round(current['RSI'], 2),
                'volume': int(current['Volume']),
                'volume_avg': int(current['Volume_MA']) if not pd.isna(current['Volume_MA']) else 0,
                'trend_confirmed': trend_confirmed,
                'price_near_ema21': price_near_ema21,
                'rsi_in_range': rsi_in_range,
                'volume_above_avg': volume_above_avg,
                'hammer': hammer,
                'bullish_engulfing': bullish_engulfing,
                'doji': doji,
                'reversal_candle': reversal_candle,
                'setup_a': setup_a,
                'momentum': momentum_score,
                'ema21_distance_pct': round(((current_price - current['EMA_21']) / current['EMA_21']) * 100, 2)
            }

        except Exception as e:
            return None

    def calculate_momentum(self, current, hourly_data):
        """Calculate momentum score"""
        try:
            rsi = current['RSI']
            vol_ratio = current['Volume'] / current['Volume_MA'] if current['Volume_MA'] > 0 else 1

            # Price change over last few hours
            if len(hourly_data) >= 3:
                price_change = (current['Close'] - hourly_data.iloc[-3]['Close']) / hourly_data.iloc[-3]['Close']
            else:
                price_change = 0

            momentum = (
                (50 - abs(rsi - 35)) * 0.4 +
                min(vol_ratio * 20, 40) * 0.3 +
                max(price_change * 1000, -20) * 0.3
            )

            return round(max(0, min(100, momentum)), 1)
        except:
            return 0

    def is_market_hours(self):
        """Check if market is open"""
        now = datetime.now()
        if now.weekday() >= 5:  # Weekend
            return False

        hour = now.hour
        return 9 <= hour <= 16  # Simplified market hours check

    def run_scan(self):
        """Run a single scan"""
        print(f"🔍 Starting scan at {datetime.now().strftime('%H:%M:%S')}")
        print(f"📊 Analyzing {len(self.tickers)} tickers...")

        self.results = []
        scan_start = time.time()

        for i, ticker in enumerate(self.tickers):
            if i % 15 == 0:  # Progress every 15 stocks
                print(f"   Progress: {i+1}/{len(self.tickers)}")

            result = self.analyze_stock(ticker)
            if result:
                self.results.append(result)

        scan_time = time.time() - scan_start
        print(f"✅ Scan completed in {scan_time:.1f} seconds")

        # Store scan in history
        self.scan_history.append({
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'results_count': len(self.results),
            'setup_a_count': len([r for r in self.results if r['setup_a']])
        })

        return self.results

    def display_results(self):
        """Display results in Colab-friendly format"""
        if not self.results:
            print("❌ No results to display")
            return

        df = pd.DataFrame(self.results)

        # Setup A stocks
        setup_a = df[df['setup_a'] == True].sort_values('momentum', ascending=False)

        print("🎯 SETUP A QUALIFIED STOCKS")
        print("=" * 80)
        if len(setup_a) > 0:
            for _, stock in setup_a.iterrows():
                print(f"🚀 {stock['ticker']} - ${stock['price']}")
                print(f"   RSI: {stock['rsi']} | Momentum: {stock['momentum']} | EMA21 Dist: {stock['ema21_distance_pct']}%")
                print(f"   Volume: {stock['volume']:,} ({stock['volume']/stock['volume_avg']:.1f}x avg)")
                reversal_type = []
                if stock['hammer']: reversal_type.append("Hammer")
                if stock['bullish_engulfing']: reversal_type.append("Bullish Engulfing")
                if stock['doji']: reversal_type.append("Doji")
                print(f"   Pattern: {', '.join(reversal_type) if reversal_type else 'Other'}")
                print()
        else:
            print("No stocks currently meet all Setup A criteria")

        # High momentum stocks
        high_momentum = df[df['momentum'] > 50].sort_values('momentum', ascending=False)

        print("🚀 HIGH MOMENTUM STOCKS (Top 10)")
        print("=" * 60)
        for _, stock in high_momentum.head(10).iterrows():
            status = "🎯" if stock['setup_a'] else "📈"
            print(f"{status} {stock['ticker']}: ${stock['price']} | RSI: {stock['rsi']} | Momentum: {stock['momentum']}")

        # Watch zone stocks
        watch_zone = df[(df['rsi_in_range']) & (df['price_near_ema21']) &
                       (df['trend_confirmed']) & (~df['setup_a'])]

        if len(watch_zone) > 0:
            print(f"\n👀 WATCH ZONE STOCKS ({len(watch_zone)})")
            print("=" * 50)
            for _, stock in watch_zone.iterrows():
                print(f"⏰ {stock['ticker']}: ${stock['price']} | RSI: {stock['rsi']} | Momentum: {stock['momentum']}")

        # Summary
        print(f"\n📈 SUMMARY")
        print("=" * 30)
        print(f"Total analyzed: {len(df)}")
        print(f"Setup A qualified: {len(setup_a)}")
        print(f"Trend confirmed: {len(df[df['trend_confirmed']])}")
        print(f"High momentum (>50): {len(high_momentum)}")
        print(f"Average RSI: {df['rsi'].mean():.1f}")

        return df

    def create_visualization(self, df):
        """Create interactive charts"""
        if df is None or len(df) == 0:
            return

        # RSI Distribution
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('RSI Distribution', 'Momentum vs RSI', 'Setup A Stocks', 'Volume Analysis'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}]]
        )

        # RSI histogram
        fig.add_trace(
            go.Histogram(x=df['rsi'], nbinsx=20, name='RSI Distribution'),
            row=1, col=1
        )

        # Momentum vs RSI scatter
        colors = ['red' if setup else 'blue' for setup in df['setup_a']]
        fig.add_trace(
            go.Scatter(x=df['rsi'], y=df['momentum'], mode='markers',
                      text=df['ticker'], name='Stocks',
                      marker=dict(color=colors)),
            row=1, col=2
        )

        # Setup A stocks
        setup_a_stocks = df[df['setup_a'] == True]
        if len(setup_a_stocks) > 0:
            fig.add_trace(
                go.Bar(x=setup_a_stocks['ticker'], y=setup_a_stocks['momentum'],
                       name='Setup A Momentum'),
                row=2, col=1
            )

        # Volume ratio
        df['volume_ratio'] = df['volume'] / df['volume_avg']
        fig.add_trace(
            go.Scatter(x=df['ticker'], y=df['volume_ratio'], mode='markers',
                      name='Volume Ratio', text=df['ticker']),
            row=2, col=2
        )

        fig.update_layout(height=800, title_text="Technical Analysis Dashboard")
        fig.show()

    def save_results(self, df):
        """Save results and download"""
        if df is None or len(df) == 0:
            return

        # Save to CSV
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"scan_results_{timestamp}.csv"
        df.to_csv(filename, index=False)

        # Create summary report
        setup_a = df[df['setup_a'] == True]
        report = f"""
TECHNICAL SCAN REPORT - {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
=================================================================

SETUP A QUALIFIED: {len(setup_a)} stocks
{setup_a[['ticker', 'price', 'rsi', 'momentum']].to_string(index=False) if len(setup_a) > 0 else 'None'}

HIGH MOMENTUM (>60): {len(df[df['momentum'] > 60])} stocks
TREND CONFIRMED: {len(df[df['trend_confirmed']])} stocks
AVERAGE RSI: {df['rsi'].mean():.1f}

ALERTS:
{chr(10).join(self.alerts) if self.alerts else 'No new alerts'}
        """

        with open(f"report_{timestamp}.txt", 'w') as f:
            f.write(report)

        print(f"📁 Results saved: {filename}")
        print("💾 Download files:")
        files.download(filename)
        files.download(f"report_{timestamp}.txt")

    def start_monitoring(self, duration_hours=8):
        """Start monitoring for specified duration"""
        print(f"🚀 Starting {duration_hours}-hour monitoring")
        print("=" * 50)

        end_time = datetime.now() + timedelta(hours=duration_hours)
        scan_count = 0

        while datetime.now() < end_time:
            clear_output(wait=True)

            print(f"📊 HOURLY STOCK SCANNER - Scan #{scan_count + 1}")
            print(f"⏰ Time: {datetime.now().strftime('%H:%M:%S')}")
            print(f"🎯 Monitoring until: {end_time.strftime('%H:%M:%S')}")
            print("=" * 60)

            if self.is_market_hours():
                # Run scan
                self.run_scan()
                df = self.display_results()

                # Create visualization every 3rd scan
                if scan_count % 3 == 0 and df is not None:
                    self.create_visualization(df)

                scan_count += 1

                # Save results every 5 scans
                if scan_count % 5 == 0:
                    self.save_results(df)

                print(f"\n⏳ Next scan in 1 hour...")
                time.sleep(3600)  # Wait 1 hour
            else:
                print("🌙 Market closed - waiting 30 minutes...")
                time.sleep(1800)  # Wait 30 minutes when market closed

# Initialize scanner
tickers = ["AAPL","MSFT","GOOGL","GOOG","AMZN","TSLA","META","NVDA","PYPL","INTC",
           "CSCO","NFLX","PEP","ADBE","AVGO","TXN","QCOM","INTU","AMGN","MDLZ",
           "SBUX","ISRG","GILD","REGN","VRTX","CSX","ADP","ILMN","EXC","BIIB",
           "LULU","IDXX","CTAS","ROST","MAR","MNST","KLAC","HOOD","CTSH","MU",
           "WBA","JD","BIDU","EXPE","CHTR","EA","FAST","HUM","INCY","KDP","KHC",
           "LRCX","MCHP","MRVL","MTCH","MSCI","NTES","NXPI","ODFL","ORLY","PAYX",
           "PCAR","PDD","SNPS","TTWO","VRSK","WBD","WDAY","XEL","ZS"]

print("🔧 Setting up Colab Stock Scanner...")
scanner = ColabTechnicalScanner(tickers)

print("""
🚀 COLAB STOCK SCANNER READY!

USAGE OPTIONS:

1️⃣ SINGLE SCAN:
   results_df = scanner.run_scan()
   scanner.display_results()

2️⃣ START MONITORING (8 hours):
   scanner.start_monitoring(duration_hours=8)

3️⃣ CUSTOM SCAN WITH VISUALIZATION:
   results_df = scanner.run_scan()
   df = scanner.display_results()
   scanner.create_visualization(df)
   scanner.save_results(df)

Ready to start scanning! 📊
""")

# Uncomment ONE of these to start:
# scanner.start_monitoring(duration_hours=4)  # Monitor for 4 hours
# results_df = scanner.run_scan(); df = scanner.display_results()  # Single scan

🔧 Setting up Colab Stock Scanner...

🚀 COLAB STOCK SCANNER READY!

USAGE OPTIONS:

1️⃣ SINGLE SCAN:
   results_df = scanner.run_scan()
   scanner.display_results()

2️⃣ START MONITORING (8 hours):
   scanner.start_monitoring(duration_hours=8)

3️⃣ CUSTOM SCAN WITH VISUALIZATION:
   results_df = scanner.run_scan()
   df = scanner.display_results()
   scanner.create_visualization(df)
   scanner.save_results(df)

Ready to start scanning! 📊



In [5]:
results_df = scanner.run_scan()
df = scanner.display_results()

🔍 Starting scan at 22:18:01
📊 Analyzing 70 tickers...
   Progress: 1/70
   Progress: 16/70
   Progress: 31/70
   Progress: 46/70
   Progress: 61/70
✅ Scan completed in 24.5 seconds
❌ No results to display




---––––––––––––––––––––––––––––––––––––––––––––-



## 15 **min**

In [6]:
# Google Colab Stock Scanner - Hourly Technical Analysis
# Run this cell to install required packages
!pip install yfinance pandas numpy schedule plotly

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import json
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output, HTML
import threading
from google.colab import files
warnings.filterwarnings('ignore')

class ColabTechnicalScanner:
    def __init__(self, tickers):
        self.tickers = list(set(tickers))  # Remove duplicates
        self.results = []
        self.previous_results = {}
        self.alerts = []
        self.scan_history = []
        self.is_running = False

    def calculate_ema(self, prices, period):
        """Calculate Exponential Moving Average"""
        return prices.ewm(span=period).mean()

    def calculate_rsi(self, prices, period=14):
        """Calculate RSI"""
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    def is_hammer(self, open_price, high, low, close):
        """Identify Hammer candlestick pattern"""
        body = abs(close - open_price)
        upper_shadow = high - max(open_price, close)
        lower_shadow = min(open_price, close) - low

        return (lower_shadow > 2 * body and
                upper_shadow < body and
                body > 0)

    def is_bullish_engulfing(self, prev_open, prev_close, curr_open, curr_close):
        """Identify Bullish Engulfing pattern"""
        prev_bearish = prev_close < prev_open
        curr_bullish = curr_close > curr_open
        engulfing = curr_open < prev_close and curr_close > prev_open

        return prev_bearish and curr_bullish and engulfing

    def is_doji(self, open_price, close, high, low):
        """Identify Doji pattern"""
        body = abs(close - open_price)
        total_range = high - low
        return body <= (total_range * 0.1) if total_range > 0 else False

    def get_market_data(self, ticker):
        """Get market data optimized for 15-minute scanning"""
        try:
            stock = yf.Ticker(ticker)

            # Get intraday data (last 5 days, 1-minute intervals for precision)
            intraday = stock.history(period="5d", interval="1m")

            # Get daily data for EMAs
            daily = stock.history(period="6mo", interval="1d")

            return intraday, daily
        except Exception as e:
            return None, None

    def analyze_stock(self, ticker):
        """Analyze individual stock"""
        try:
            intraday, daily = self.get_market_data(ticker)

            if intraday is None or daily is None or len(daily) < 50:
                return None

            # Calculate daily EMAs
            daily['EMA_21'] = self.calculate_ema(daily['Close'], 21)
            daily['EMA_50'] = self.calculate_ema(daily['Close'], 50)
            daily['EMA_200'] = self.calculate_ema(daily['Close'], 200)

            # Resample intraday to hourly
            hourly = intraday.resample('1H').agg({
                'Open': 'first',
                'High': 'max',
                'Low': 'min',
                'Close': 'last',
                'Volume': 'sum'
            }).dropna()

            if len(hourly) < 20:
                return None

            # Calculate hourly indicators
            hourly['EMA_21'] = self.calculate_ema(hourly['Close'], 21)
            hourly['RSI'] = self.calculate_rsi(hourly['Close'], 14)
            hourly['Volume_MA'] = hourly['Volume'].rolling(window=20).mean()

            # Current values
            current = hourly.iloc[-1]
            prev = hourly.iloc[-2] if len(hourly) > 1 else current
            daily_current = daily.iloc[-1]

            current_price = current['Close']

            # Trend confirmation
            trend_confirmed = (daily_current['EMA_50'] > daily_current['EMA_200'] and
                             current_price > daily_current['EMA_21'])

            # Setup criteria
            price_near_ema21 = abs(current_price - current['EMA_21']) / current['EMA_21'] <= 0.03
            rsi_in_range = 25 <= current['RSI'] <= 45
            volume_above_avg = current['Volume'] >= (1.3 * current['Volume_MA'])

            # Reversal patterns
            hammer = self.is_hammer(current['Open'], current['High'], current['Low'], current['Close'])
            bullish_engulfing = self.is_bullish_engulfing(prev['Open'], prev['Close'],
                                                         current['Open'], current['Close'])
            doji = self.is_doji(current['Open'], current['Close'], current['High'], current['Low'])
            reversal_candle = hammer or bullish_engulfing or (doji and current['RSI'] < 40)

            # Setup A qualification
            setup_a = (price_near_ema21 and rsi_in_range and
                      reversal_candle and volume_above_avg and trend_confirmed)

            # Momentum score
            momentum_score = self.calculate_momentum(current, hourly)

            return {
                'ticker': ticker,
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'price': round(current_price, 2),
                'ema_21': round(current['EMA_21'], 2),
                'daily_ema_50': round(daily_current['EMA_50'], 2),
                'daily_ema_200': round(daily_current['EMA_200'], 2),
                'rsi': round(current['RSI'], 2),
                'volume': int(current['Volume']),
                'volume_avg': int(current['Volume_MA']) if not pd.isna(current['Volume_MA']) else 0,
                'trend_confirmed': trend_confirmed,
                'price_near_ema21': price_near_ema21,
                'rsi_in_range': rsi_in_range,
                'volume_above_avg': volume_above_avg,
                'hammer': hammer,
                'bullish_engulfing': bullish_engulfing,
                'doji': doji,
                'reversal_candle': reversal_candle,
                'setup_a': setup_a,
                'momentum': momentum_score,
                'ema21_distance_pct': round(((current_price - current['EMA_21']) / current['EMA_21']) * 100, 2)
            }

        except Exception as e:
            return None

    def calculate_momentum(self, current, hourly_data):
        """Calculate momentum score"""
        try:
            rsi = current['RSI']
            vol_ratio = current['Volume'] / current['Volume_MA'] if current['Volume_MA'] > 0 else 1

            # Price change over last few hours
            if len(hourly_data) >= 3:
                price_change = (current['Close'] - hourly_data.iloc[-3]['Close']) / hourly_data.iloc[-3]['Close']
            else:
                price_change = 0

            momentum = (
                (50 - abs(rsi - 35)) * 0.4 +
                min(vol_ratio * 20, 40) * 0.3 +
                max(price_change * 1000, -20) * 0.3
            )

            return round(max(0, min(100, momentum)), 1)
        except:
            return 0

    def is_market_hours(self):
        """Check if market is open"""
        now = datetime.now()
        if now.weekday() >= 5:  # Weekend
            return False

        hour = now.hour
        return 9 <= hour <= 16  # Simplified market hours check

    def run_scan(self):
        """Run a single scan"""
        print(f"🔍 Starting scan at {datetime.now().strftime('%H:%M:%S')}")
        print(f"📊 Analyzing {len(self.tickers)} tickers...")

        self.results = []
        scan_start = time.time()

        for i, ticker in enumerate(self.tickers):
            if i % 15 == 0:  # Progress every 15 stocks
                print(f"   Progress: {i+1}/{len(self.tickers)}")

            result = self.analyze_stock(ticker)
            if result:
                self.results.append(result)

        scan_time = time.time() - scan_start
        print(f"✅ Scan completed in {scan_time:.1f} seconds")

        # Store scan in history
        self.scan_history.append({
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'results_count': len(self.results),
            'setup_a_count': len([r for r in self.results if r['setup_a']])
        })

        return self.results

    def display_results(self):
        """Display results in Colab-friendly format"""
        if not self.results:
            print("❌ No results to display")
            return

        df = pd.DataFrame(self.results)

        # Setup A stocks
        setup_a = df[df['setup_a'] == True].sort_values('momentum', ascending=False)

        print("🎯 SETUP A QUALIFIED STOCKS")
        print("=" * 80)
        if len(setup_a) > 0:
            for _, stock in setup_a.iterrows():
                print(f"🚀 {stock['ticker']} - ${stock['price']}")
                print(f"   RSI: {stock['rsi']} | Momentum: {stock['momentum']} | EMA21 Dist: {stock['ema21_distance_pct']}%")
                print(f"   Volume: {stock['volume']:,} ({stock['volume']/stock['volume_avg']:.1f}x avg)")
                reversal_type = []
                if stock['hammer']: reversal_type.append("Hammer")
                if stock['bullish_engulfing']: reversal_type.append("Bullish Engulfing")
                if stock['doji']: reversal_type.append("Doji")
                print(f"   Pattern: {', '.join(reversal_type) if reversal_type else 'Other'}")
                print()
        else:
            print("No stocks currently meet all Setup A criteria")

        # High momentum stocks
        high_momentum = df[df['momentum'] > 50].sort_values('momentum', ascending=False)

        print("🚀 HIGH MOMENTUM STOCKS (Top 10)")
        print("=" * 60)
        for _, stock in high_momentum.head(10).iterrows():
            status = "🎯" if stock['setup_a'] else "📈"
            print(f"{status} {stock['ticker']}: ${stock['price']} | RSI: {stock['rsi']} | Momentum: {stock['momentum']}")

        # Watch zone stocks
        watch_zone = df[(df['rsi_in_range']) & (df['price_near_ema21']) &
                       (df['trend_confirmed']) & (~df['setup_a'])]

        if len(watch_zone) > 0:
            print(f"\n👀 WATCH ZONE STOCKS ({len(watch_zone)})")
            print("=" * 50)
            for _, stock in watch_zone.iterrows():
                print(f"⏰ {stock['ticker']}: ${stock['price']} | RSI: {stock['rsi']} | Momentum: {stock['momentum']}")

        # Summary
        print(f"\n📈 SUMMARY")
        print("=" * 30)
        print(f"Total analyzed: {len(df)}")
        print(f"Setup A qualified: {len(setup_a)}")
        print(f"Trend confirmed: {len(df[df['trend_confirmed']])}")
        print(f"High momentum (>50): {len(high_momentum)}")
        print(f"Average RSI: {df['rsi'].mean():.1f}")

        return df

    def create_visualization(self, df):
        """Create interactive charts"""
        if df is None or len(df) == 0:
            return

        # RSI Distribution
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('RSI Distribution', 'Momentum vs RSI', 'Setup A Stocks', 'Volume Analysis'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}]]
        )

        # RSI histogram
        fig.add_trace(
            go.Histogram(x=df['rsi'], nbinsx=20, name='RSI Distribution'),
            row=1, col=1
        )

        # Momentum vs RSI scatter
        colors = ['red' if setup else 'blue' for setup in df['setup_a']]
        fig.add_trace(
            go.Scatter(x=df['rsi'], y=df['momentum'], mode='markers',
                      text=df['ticker'], name='Stocks',
                      marker=dict(color=colors)),
            row=1, col=2
        )

        # Setup A stocks
        setup_a_stocks = df[df['setup_a'] == True]
        if len(setup_a_stocks) > 0:
            fig.add_trace(
                go.Bar(x=setup_a_stocks['ticker'], y=setup_a_stocks['momentum'],
                       name='Setup A Momentum'),
                row=2, col=1
            )

        # Volume ratio
        df['volume_ratio'] = df['volume'] / df['volume_avg']
        fig.add_trace(
            go.Scatter(x=df['ticker'], y=df['volume_ratio'], mode='markers',
                      name='Volume Ratio', text=df['ticker']),
            row=2, col=2
        )

        fig.update_layout(height=800, title_text="Technical Analysis Dashboard")
        fig.show()

    def save_results(self, df):
        """Save results and download"""
        if df is None or len(df) == 0:
            return

        # Save to CSV
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"scan_results_{timestamp}.csv"
        df.to_csv(filename, index=False)

        # Create summary report
        setup_a = df[df['setup_a'] == True]
        report = f"""
TECHNICAL SCAN REPORT - {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
=================================================================

SETUP A QUALIFIED: {len(setup_a)} stocks
{setup_a[['ticker', 'price', 'rsi', 'momentum']].to_string(index=False) if len(setup_a) > 0 else 'None'}

HIGH MOMENTUM (>60): {len(df[df['momentum'] > 60])} stocks
TREND CONFIRMED: {len(df[df['trend_confirmed']])} stocks
AVERAGE RSI: {df['rsi'].mean():.1f}

ALERTS:
{chr(10).join(self.alerts) if self.alerts else 'No new alerts'}
        """

        with open(f"report_{timestamp}.txt", 'w') as f:
            f.write(report)

        print(f"📁 Results saved: {filename}")
        print("💾 Download files:")
        files.download(filename)
        files.download(f"report_{timestamp}.txt")

    def start_monitoring(self, duration_hours=8):
        """Start monitoring for specified duration"""
        print(f"🚀 Starting {duration_hours}-hour monitoring")
        print("=" * 50)

        end_time = datetime.now() + timedelta(hours=duration_hours)
        scan_count = 0

        while datetime.now() < end_time:
            clear_output(wait=True)

            print(f"📊 HOURLY STOCK SCANNER - Scan #{scan_count + 1}")
            print(f"⏰ Time: {datetime.now().strftime('%H:%M:%S')}")
            print(f"🎯 Monitoring until: {end_time.strftime('%H:%M:%S')}")
            print("=" * 60)

            if self.is_market_hours():
                # Run scan
                self.run_scan()
                df = self.display_results()

                # Create visualization every 3rd scan
                if scan_count % 3 == 0 and df is not None:
                    self.create_visualization(df)

                scan_count += 1

                # Save results every 5 scans
                if scan_count % 5 == 0:
                    self.save_results(df)

                print(f"\n⏳ Next scan in 1 hour...")
                time.sleep(3600)  # Wait 1 hour
            else:
                print("🌙 Market closed - waiting 30 minutes...")
                time.sleep(1800)  # Wait 30 minutes when market closed

# Initialize scanner
tickers = ["AAPL","MSFT","GOOGL","GOOG","AMZN","TSLA","META","NVDA","PYPL","INTC",
           "CSCO","NFLX","PEP","ADBE","AVGO","TXN","QCOM","INTU","AMGN","MDLZ",
           "SBUX","ISRG","GILD","REGN","VRTX","CSX","ADP","ILMN","EXC","BIIB",
           "LULU","IDXX","CTAS","ROST","MAR","MNST","KLAC","CTSH","MU",
           "WBA","JD","BIDU","EXPE","CHTR","EA","FAST","HUM","INCY","KDP","KHC",
           "LRCX","MCHP","MRVL","MTCH","MSCI","NTES","NXPI","ODFL","ORLY","PAYX",
           "PCAR","PDD","SNPS","TTWO","VRSK","WBD","WDAY","XEL","ZS"]

print("🔧 Setting up Colab Stock Scanner...")
scanner = ColabTechnicalScanner(tickers)

print("""
🚀 COLAB STOCK SCANNER READY!

USAGE OPTIONS:

1️⃣ SINGLE SCAN:
   results_df = scanner.run_scan()
   scanner.display_results()

2️⃣ START MONITORING (8 hours):
   scanner.start_monitoring(duration_hours=8)

3️⃣ CUSTOM SCAN WITH VISUALIZATION:
   results_df = scanner.run_scan()
   df = scanner.display_results()
   scanner.create_visualization(df)
   scanner.save_results(df)

Ready to start scanning! 📊
""")

# Uncomment ONE of these to start:
# scanner.start_monitoring(duration_hours=4)  # Monitor for 4 hours
# results_df = scanner.run_scan(); df = scanner.display_results()  # Single scan

🔧 Setting up Colab Stock Scanner...

🚀 COLAB STOCK SCANNER READY!

USAGE OPTIONS:

1️⃣ SINGLE SCAN:
   results_df = scanner.run_scan()
   scanner.display_results()

2️⃣ START MONITORING (8 hours):
   scanner.start_monitoring(duration_hours=8)

3️⃣ CUSTOM SCAN WITH VISUALIZATION:
   results_df = scanner.run_scan()
   df = scanner.display_results()
   scanner.create_visualization(df)
   scanner.save_results(df)

Ready to start scanning! 📊



In [7]:
results_df = scanner.run_scan()
df = scanner.display_results()

🔍 Starting scan at 22:18:43
📊 Analyzing 69 tickers...
   Progress: 1/69
   Progress: 16/69
   Progress: 31/69
   Progress: 46/69
   Progress: 61/69
✅ Scan completed in 29.3 seconds
🎯 SETUP A QUALIFIED STOCKS
🚀 INCY - $84.79
   RSI: 27.12 | Momentum: 28.9 | EMA21 Dist: -0.77%
   Volume: 368,631 (2.5x avg)
   Pattern: Bullish Engulfing

🚀 HIGH MOMENTUM STOCKS (Top 10)

👀 WATCH ZONE STOCKS (3)
⏰ ORLY: $102.48 | RSI: 44.35 | Momentum: 27.2
⏰ BIIB: $136.34 | RSI: 30.73 | Momentum: 26.9
⏰ PEP: $148.22 | RSI: 44.37 | Momentum: 27.9

📈 SUMMARY
Total analyzed: 69
Setup A qualified: 1
Trend confirmed: 36
High momentum (>50): 0
Average RSI: 54.2


⏰.   ⏰ #######%%%%%%%%%%&&&&&&&&&(((((((((()))))))))))@@@@@@@@@@@@@@@


# **Daily **

Perfect! I've significantly enhanced the candlestick pattern recognition with 10 powerful bullish reversal patterns! 🚀
🔥 New Powerful Patterns Added:
Tier 3 - MOST POWERFUL (🔥):

Bullish Engulfing - Enhanced with size requirements
Morning Star - 3-candle reversal pattern
Three White Soldiers - Strong uptrend continuation
Abandoned Baby Bullish - Rare gap reversal pattern

Tier 2 - STRONG (⚡):

Hammer - Enhanced with strict criteria
Piercing Pattern - Gap-down reversal
Bullish Harami - Contained reversal pattern
Inverted Hammer - Top reversal signal

Tier 1 - MODERATE (📈):

Dragonfly Doji - Long lower shadow
Regular Doji - Only when RSI < 35

🎯 Key Enhancements:
Smart Pattern Scoring:

Strength 3/3: Most powerful patterns (Morning Star, Bullish Engulfing, etc.)
Strength 2/3: Strong patterns (Hammer, Piercing, etc.)
Strength 1/3: Moderate patterns (only when oversold)

Enhanced Setup A Criteria:

Now requires Strength 2/3 or higher patterns
Shows specific patterns found for each stock
Pattern breakdown summary showing frequency

Better Visual Display:

🔥 for Strength 3 patterns
⚡ for Strength 2 patterns
📈 for Strength 1 patterns
Lists exact patterns found per stock

New Report Sections:

Powerful Reversal Patterns - Strength 3 signals
Pattern Breakdown - Shows which patterns are most common
Enhanced summaries with pattern strength stats

🚀 What This Means:
Your Setup A signals are now much more reliable because:

Only accepts high-probability reversal patterns
Filters out weak signals
Prioritizes the most powerful candlestick formations
Gives you exact pattern names for each opportunity

The scanner will now catch Morning Stars at support, Bullish Engulfing patterns, and other institutional-quality reversal signals! 📊⚡
Ready to catch those powerful reversals! 🎯

In [12]:
# Google Colab Stock Scanner - Hourly Technical Analysis
# Run this cell to install required packages
!pip install yfinance pandas numpy schedule plotly

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import json
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output, HTML
import threading
from google.colab import files
warnings.filterwarnings('ignore')

class ColabTechnicalScanner:
    def __init__(self, tickers):
        self.tickers = list(set(tickers))  # Remove duplicates
        self.results = []
        self.previous_results = {}
        self.alerts = []
        self.scan_history = []
        self.is_running = False

    def calculate_ema(self, prices, period):
        """Calculate Exponential Moving Average"""
        return prices.ewm(span=period).mean()

    def calculate_rsi(self, prices, period=14):
        """Calculate RSI"""
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    def is_hammer(self, open_price, high, low, close):
        """Identify Hammer candlestick pattern"""
        body = abs(close - open_price)
        upper_shadow = high - max(open_price, close)
        lower_shadow = min(open_price, close) - low
        total_range = high - low

        if total_range == 0:
            return False

        return (lower_shadow > 2 * body and
                upper_shadow < body and
                body > 0 and
                lower_shadow > 0.6 * total_range)  # Strong hammer criteria

    def is_inverted_hammer(self, open_price, high, low, close):
        """Identify Inverted Hammer pattern"""
        body = abs(close - open_price)
        upper_shadow = high - max(open_price, close)
        lower_shadow = min(open_price, close) - low
        total_range = high - low

        if total_range == 0:
            return False

        return (upper_shadow > 2 * body and
                lower_shadow < body and
                body > 0 and
                upper_shadow > 0.6 * total_range)

    def is_bullish_engulfing(self, prev_open, prev_close, curr_open, curr_close):
        """Identify Bullish Engulfing pattern"""
        prev_bearish = prev_close < prev_open
        curr_bullish = curr_close > curr_open
        engulfing = curr_open <= prev_close and curr_close > prev_open
        significant_body = abs(curr_close - curr_open) > abs(prev_close - prev_open) * 1.1

        return prev_bearish and curr_bullish and engulfing and significant_body

    def is_piercing_pattern(self, prev_open, prev_high, prev_low, prev_close,
                           curr_open, curr_high, curr_low, curr_close):
        """Identify Piercing Line pattern"""
        prev_bearish = prev_close < prev_open
        curr_bullish = curr_close > curr_open
        gap_down = curr_open < prev_low
        penetration = curr_close > (prev_open + prev_close) / 2

        return prev_bearish and curr_bullish and gap_down and penetration

    def is_morning_star(self, candles):
        """Identify Morning Star pattern (3-candle pattern)"""
        if len(candles) < 3:
            return False

        first, second, third = candles[-3], candles[-2], candles[-1]

        # First candle: Bearish
        first_bearish = first['Close'] < first['Open']
        first_body = abs(first['Close'] - first['Open'])

        # Second candle: Small body (star)
        second_body = abs(second['Close'] - second['Open'])
        second_small = second_body < first_body * 0.3
        gap_down = second['High'] < first['Close']

        # Third candle: Bullish
        third_bullish = third['Close'] > third['Open']
        third_body = abs(third['Close'] - third['Open'])
        penetration = third['Close'] > (first['Open'] + first['Close']) / 2

        return (first_bearish and second_small and gap_down and
                third_bullish and penetration and third_body > first_body * 0.6)

    def is_bullish_harami(self, prev_open, prev_high, prev_low, prev_close,
                         curr_open, curr_high, curr_low, curr_close):
        """Identify Bullish Harami pattern"""
        prev_bearish = prev_close < prev_open
        curr_bullish = curr_close > curr_open

        # Current candle contained within previous candle's body
        contained = (curr_open > prev_close and curr_open < prev_open and
                    curr_close > prev_close and curr_close < prev_open)

        prev_body = abs(prev_close - prev_open)
        curr_body = abs(curr_close - curr_open)
        significant_prev = prev_body > (prev_high - prev_low) * 0.6

        return prev_bearish and curr_bullish and contained and significant_prev

    def is_dragonfly_doji(self, open_price, high, low, close):
        """Identify Dragonfly Doji pattern"""
        body = abs(close - open_price)
        upper_shadow = high - max(open_price, close)
        lower_shadow = min(open_price, close) - low
        total_range = high - low

        if total_range == 0:
            return False

        return (body <= total_range * 0.1 and
                upper_shadow <= total_range * 0.1 and
                lower_shadow >= total_range * 0.7)

    def is_three_white_soldiers(self, candles):
        """Identify Three White Soldiers pattern"""
        if len(candles) < 3:
            return False

        last_three = candles[-3:]

        for candle in last_three:
            # All must be bullish
            if candle['Close'] <= candle['Open']:
                return False

            # Each should have reasonable body size
            body = abs(candle['Close'] - candle['Open'])
            total_range = candle['High'] - candle['Low']
            if body < total_range * 0.6:  # Strong bullish bodies
                return False

        # Each close should be higher than previous
        for i in range(1, 3):
            if last_three[i]['Close'] <= last_three[i-1]['Close']:
                return False

        # Each open should be within previous candle's body
        for i in range(1, 3):
            prev_body_mid = (last_three[i-1]['Open'] + last_three[i-1]['Close']) / 2
            if last_three[i]['Open'] < prev_body_mid:
                return False

        return True

    def is_abandoned_baby_bullish(self, candles):
        """Identify Bullish Abandoned Baby pattern"""
        if len(candles) < 3:
            return False

        first, second, third = candles[-3], candles[-2], candles[-1]

        # First: Strong bearish
        first_bearish = first['Close'] < first['Open']
        first_body = abs(first['Close'] - first['Open'])

        # Second: Doji with gaps
        second_doji = abs(second['Close'] - second['Open']) < (second['High'] - second['Low']) * 0.1
        gap_down = second['High'] < first['Low']
        gap_up = third['Low'] > second['High']

        # Third: Strong bullish
        third_bullish = third['Close'] > third['Open']
        third_body = abs(third['Close'] - third['Open'])

        return (first_bearish and second_doji and gap_down and gap_up and
                third_bullish and first_body > 0 and third_body > 0)

    def is_doji(self, open_price, close, high, low):
        """Identify Doji pattern (enhanced)"""
        body = abs(close - open_price)
        total_range = high - low

        if total_range == 0:
            return False

        return body <= (total_range * 0.1)

    def get_market_data(self, ticker):
        """Get daily market data for candlestick pattern analysis"""
        try:
            stock = yf.Ticker(ticker)

            # Get daily data (1 year for comprehensive analysis)
            daily = stock.history(period="1y", interval="1d")

            return daily
        except Exception as e:
            return None

    def analyze_stock(self, ticker):
        """Analyze individual stock using daily timeframe"""
        try:
            daily = self.get_market_data(ticker)

            if daily is None or len(daily) < 200:
                return None

            # Calculate daily EMAs and indicators
            daily['EMA_21'] = self.calculate_ema(daily['Close'], 21)
            daily['EMA_50'] = self.calculate_ema(daily['Close'], 50)
            daily['EMA_200'] = self.calculate_ema(daily['Close'], 200)
            daily['RSI'] = self.calculate_rsi(daily['Close'], 14)
            daily['Volume_MA'] = daily['Volume'].rolling(window=20).mean()

            # Current values
            current = daily.iloc[-1]
            prev = daily.iloc[-2] if len(daily) > 1 else current

            current_price = current['Close']

            # Trend confirmation
            trend_confirmed = (current['EMA_50'] > current['EMA_200'] and
                             current_price > current['EMA_21'])

            # Setup criteria (daily timeframe)
            price_near_ema21 = abs(current_price - current['EMA_21']) / current['EMA_21'] <= 0.02
            rsi_in_range = 30 <= current['RSI'] <= 40  # Original RSI range
            volume_above_avg = current['Volume'] >= (1.5 * current['Volume_MA'])  # Original volume threshold

            # Enhanced reversal pattern detection using daily candles
            candles_data = []
            if len(daily) >= 3:
                for i in range(-3, 0):
                    candles_data.append({
                        'Open': daily.iloc[i]['Open'],
                        'High': daily.iloc[i]['High'],
                        'Low': daily.iloc[i]['Low'],
                        'Close': daily.iloc[i]['Close']
                    })

            # Single candle patterns
            hammer = self.is_hammer(current['Open'], current['High'], current['Low'], current['Close'])
            inverted_hammer = self.is_inverted_hammer(current['Open'], current['High'], current['Low'], current['Close'])
            dragonfly_doji = self.is_dragonfly_doji(current['Open'], current['High'], current['Low'], current['Close'])
            regular_doji = self.is_doji(current['Open'], current['Close'], current['High'], current['Low'])

            # Two candle patterns
            bullish_engulfing = self.is_bullish_engulfing(prev['Open'], prev['Close'],
                                                         current['Open'], current['Close'])
            piercing_pattern = self.is_piercing_pattern(prev['Open'], prev['High'], prev['Low'], prev['Close'],
                                                       current['Open'], current['High'], current['Low'], current['Close'])
            bullish_harami = self.is_bullish_harami(prev['Open'], prev['High'], prev['Low'], prev['Close'],
                                                   current['Open'], current['High'], current['Low'], current['Close'])

            # Three candle patterns
            morning_star = self.is_morning_star(candles_data) if len(candles_data) >= 3 else False
            three_white_soldiers = self.is_three_white_soldiers(candles_data) if len(candles_data) >= 3 else False
            abandoned_baby_bullish = self.is_abandoned_baby_bullish(candles_data) if len(candles_data) >= 3 else False

            # Aggregate reversal signal
            powerful_patterns = [bullish_engulfing, morning_star, three_white_soldiers, abandoned_baby_bullish]
            moderate_patterns = [hammer, piercing_pattern, bullish_harami, inverted_hammer]
            weak_patterns = [dragonfly_doji, regular_doji]

            # Pattern strength scoring
            pattern_strength = 0
            if any(powerful_patterns):
                pattern_strength = 3  # Very strong
            elif any(moderate_patterns):
                pattern_strength = 2  # Strong
            elif any(weak_patterns) and current['RSI'] < 35:
                pattern_strength = 1  # Moderate (only if oversold)

            reversal_candle = pattern_strength >= 2  # Require at least strong patterns

            # Identify specific patterns found
            patterns_found = []
            if bullish_engulfing: patterns_found.append("Bullish Engulfing")
            if morning_star: patterns_found.append("Morning Star")
            if three_white_soldiers: patterns_found.append("Three White Soldiers")
            if abandoned_baby_bullish: patterns_found.append("Abandoned Baby")
            if hammer: patterns_found.append("Hammer")
            if piercing_pattern: patterns_found.append("Piercing Pattern")
            if bullish_harami: patterns_found.append("Bullish Harami")
            if inverted_hammer: patterns_found.append("Inverted Hammer")
            if dragonfly_doji: patterns_found.append("Dragonfly Doji")
            if regular_doji and current['RSI'] < 35: patterns_found.append("Doji (Oversold)")

            # Setup A qualification
            setup_a = (price_near_ema21 and rsi_in_range and
                      reversal_candle and volume_above_avg and trend_confirmed)

            # Daily momentum score
            momentum_score = self.calculate_daily_momentum(current, daily)

            return {
                'ticker': ticker,
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'price': round(current_price, 2),
                'ema_21': round(current['EMA_21'], 2),
                'ema_50': round(current['EMA_50'], 2),
                'ema_200': round(current['EMA_200'], 2),
                'rsi': round(current['RSI'], 2),
                'volume': int(current['Volume']),
                'volume_avg': int(current['Volume_MA']) if not pd.isna(current['Volume_MA']) else 0,
                'trend_confirmed': trend_confirmed,
                'price_near_ema21': price_near_ema21,
                'rsi_in_range': rsi_in_range,
                'volume_above_avg': volume_above_avg,
                'reversal_candle': reversal_candle,
                'pattern_strength': pattern_strength,
                'patterns_found': patterns_found,
                'setup_a': setup_a,
                'momentum': momentum_score,
                'ema21_distance_pct': round(((current_price - current['EMA_21']) / current['EMA_21']) * 100, 2),
                # Individual pattern flags for detailed analysis
                'bullish_engulfing': bullish_engulfing,
                'morning_star': morning_star,
                'three_white_soldiers': three_white_soldiers,
                'abandoned_baby': abandoned_baby_bullish,
                'hammer': hammer,
                'piercing_pattern': piercing_pattern,
                'bullish_harami': bullish_harami,
                'inverted_hammer': inverted_hammer,
                'dragonfly_doji': dragonfly_doji
            }

        except Exception as e:
            return None

    def calculate_daily_momentum(self, current, daily_data):
        """Calculate momentum score for daily timeframe"""
        try:
            rsi = current['RSI']
            vol_ratio = current['Volume'] / current['Volume_MA'] if current['Volume_MA'] > 0 else 1

            # Price change over last 5 days
            if len(daily_data) >= 5:
                five_day_change = (current['Close'] - daily_data.iloc[-5]['Close']) / daily_data.iloc[-5]['Close']
            else:
                five_day_change = 0

            # Price change over last 10 days
            if len(daily_data) >= 10:
                ten_day_change = (current['Close'] - daily_data.iloc[-10]['Close']) / daily_data.iloc[-10]['Close']
            else:
                ten_day_change = 0

            # Relative strength vs EMA
            ema_strength = (current['Close'] - current['EMA_21']) / current['EMA_21'] if current['EMA_21'] > 0 else 0

            momentum = (
                (50 - abs(rsi - 35)) * 0.3 +  # RSI near 35 is ideal for pullback
                min(vol_ratio * 20, 30) * 0.25 +  # Volume boost
                max(five_day_change * 500, -15) * 0.25 +  # 5-day momentum
                max(ten_day_change * 300, -10) * 0.1 +  # 10-day momentum
                max(ema_strength * 100, -10) * 0.1  # EMA relationship
            )

            return round(max(0, min(100, momentum)), 1)
        except:
            return 0

    def is_market_hours(self):
        """Check if market is open"""
        now = datetime.now()
        if now.weekday() >= 5:  # Weekend
            return False

        hour = now.hour
        return 9 <= hour <= 16  # Simplified market hours check

    def run_scan(self):
        """Run a single daily scan"""
        print(f"🔍 Starting daily scan at {datetime.now().strftime('%H:%M:%S')}")
        print(f"📊 Analyzing {len(self.tickers)} tickers with daily candlestick patterns...")

        self.results = []
        scan_start = time.time()

        for i, ticker in enumerate(self.tickers):
            if i % 10 == 0:  # Progress every 10 stocks
                print(f"   Progress: {i+1}/{len(self.tickers)}")

            result = self.analyze_stock(ticker)
            if result:
                self.results.append(result)

        scan_time = time.time() - scan_start
        print(f"✅ Daily scan completed in {scan_time:.1f} seconds")

        # Store scan in history
        self.scan_history.append({
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'results_count': len(self.results),
            'setup_a_count': len([r for r in self.results if r['setup_a']])
        })

        return self.results

    def display_results(self):
        """Display results in Colab-friendly format"""
        if not self.results:
            print("❌ No results to display")
            return

        df = pd.DataFrame(self.results)

        # Setup A stocks
        setup_a = df[df['setup_a'] == True].sort_values('momentum', ascending=False)

        print("🎯 SETUP A QUALIFIED STOCKS")
        print("=" * 80)
        if len(setup_a) > 0:
            for _, stock in setup_a.iterrows():
                patterns_str = ", ".join(stock['patterns_found']) if stock['patterns_found'] else "Other"
                strength_emoji = "🔥" if stock['pattern_strength'] == 3 else "⚡" if stock['pattern_strength'] == 2 else "📈"

                print(f"{strength_emoji} {stock['ticker']} - ${stock['price']}")
                print(f"   RSI: {stock['rsi']} | Momentum: {stock['momentum']} | EMA21 Dist: {stock['ema21_distance_pct']}%")
                print(f"   Volume: {stock['volume']:,} ({stock['volume']/stock['volume_avg']:.1f}x avg)")
                print(f"   🎯 Pattern: {patterns_str} (Strength: {stock['pattern_strength']}/3)")
                print()
        else:
            print("No stocks currently meet all Setup A criteria")

        # High momentum stocks
        high_momentum = df[df['momentum'] > 50].sort_values('momentum', ascending=False)

        print("🚀 HIGH MOMENTUM STOCKS (Top 15)")
        print("=" * 60)
        for _, stock in high_momentum.head(15).iterrows():
            status = "🎯" if stock['setup_a'] else "📈"
            print(f"{status} {stock['ticker']}: ${stock['price']} | RSI: {stock['rsi']} | Momentum: {stock['momentum']}")

        # Powerful pattern alerts
        powerful_patterns = df[(df['pattern_strength'] >= 3) & (df['trend_confirmed'] == True)]
        if len(powerful_patterns) > 0:
            print(f"\n🔥 POWERFUL REVERSAL PATTERNS ({len(powerful_patterns)})")
            print("=" * 60)
            for _, stock in powerful_patterns.iterrows():
                patterns_str = ", ".join(stock['patterns_found'])
                print(f"🔥 {stock['ticker']}: ${stock['price']} | RSI: {stock['rsi']} | Pattern: {patterns_str}")

        # Pattern breakdown
        pattern_summary = {}
        for _, stock in df.iterrows():
            for pattern in stock['patterns_found']:
                if pattern not in pattern_summary:
                    pattern_summary[pattern] = 0
                pattern_summary[pattern] += 1

        if pattern_summary:
            print(f"\n📊 PATTERN BREAKDOWN:")
            print("=" * 40)
            for pattern, count in sorted(pattern_summary.items(), key=lambda x: x[1], reverse=True):
                print(f"   {pattern}: {count} stocks")

        # Watch zone stocks
        watch_zone = df[(df['rsi_in_range']) & (df['price_near_ema21']) &
                       (df['trend_confirmed']) & (~df['setup_a'])]

        if len(watch_zone) > 0:
            print(f"\n👀 WATCH ZONE STOCKS ({len(watch_zone)})")
            print("=" * 50)
            for _, stock in watch_zone.iterrows():
                patterns_str = ", ".join(stock['patterns_found']) if stock['patterns_found'] else "Waiting for signal"
                print(f"⏰ {stock['ticker']}: ${stock['price']} | RSI: {stock['rsi']} | {patterns_str}")

        # Summary
        print(f"\n📈 SUMMARY")
        print("=" * 30)
        print(f"Total analyzed: {len(df)}")
        print(f"Setup A qualified: {len(setup_a)}")
        print(f"Powerful patterns (3/3): {len(df[df['pattern_strength'] >= 3])}")
        print(f"Strong patterns (2/3): {len(df[df['pattern_strength'] >= 2])}")
        print(f"Trend confirmed: {len(df[df['trend_confirmed']])}")
        print(f"High momentum (>50): {len(high_momentum)}")
        print(f"Average RSI: {df['rsi'].mean():.1f}")

        return df

    def create_visualization(self, df):
        """Create interactive charts"""
        if df is None or len(df) == 0:
            return

        # RSI Distribution
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('RSI Distribution', 'Momentum vs RSI', 'Setup A Stocks', 'Volume Analysis'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}]]
        )

        # RSI histogram
        fig.add_trace(
            go.Histogram(x=df['rsi'], nbinsx=20, name='RSI Distribution'),
            row=1, col=1
        )

        # Momentum vs RSI scatter
        colors = ['red' if setup else 'blue' for setup in df['setup_a']]
        fig.add_trace(
            go.Scatter(x=df['rsi'], y=df['momentum'], mode='markers',
                      text=df['ticker'], name='Stocks',
                      marker=dict(color=colors)),
            row=1, col=2
        )

        # Setup A stocks
        setup_a_stocks = df[df['setup_a'] == True]
        if len(setup_a_stocks) > 0:
            fig.add_trace(
                go.Bar(x=setup_a_stocks['ticker'], y=setup_a_stocks['momentum'],
                       name='Setup A Momentum'),
                row=2, col=1
            )

        # Volume ratio
        df['volume_ratio'] = df['volume'] / df['volume_avg']
        fig.add_trace(
            go.Scatter(x=df['ticker'], y=df['volume_ratio'], mode='markers',
                      name='Volume Ratio', text=df['ticker']),
            row=2, col=2
        )

        fig.update_layout(height=800, title_text="Technical Analysis Dashboard")
        fig.show()

    def save_results(self, df):
        """Save results and download"""
        if df is None or len(df) == 0:
            return

        # Save to CSV
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"scan_results_{timestamp}.csv"
        df.to_csv(filename, index=False)

        # Create summary report
        setup_a = df[df['setup_a'] == True]
        report = f"""
TECHNICAL SCAN REPORT - {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
=================================================================

SETUP A QUALIFIED: {len(setup_a)} stocks
{setup_a[['ticker', 'price', 'rsi', 'momentum']].to_string(index=False) if len(setup_a) > 0 else 'None'}

HIGH MOMENTUM (>60): {len(df[df['momentum'] > 60])} stocks
TREND CONFIRMED: {len(df[df['trend_confirmed']])} stocks
AVERAGE RSI: {df['rsi'].mean():.1f}

ALERTS:
{chr(10).join(self.alerts) if self.alerts else 'No new alerts'}
        """

        with open(f"report_{timestamp}.txt", 'w') as f:
            f.write(report)

        print(f"📁 Results saved: {filename}")
        print("💾 Download files:")
        files.download(filename)
        files.download(f"report_{timestamp}.txt")

    def start_monitoring(self, duration_hours=8):
        """Start daily monitoring (check multiple times per day)"""
        print(f"🚀 Starting {duration_hours}-hour monitoring with daily analysis")
        print("📈 Professional daily candlestick pattern recognition!")
        print("=" * 70)

        end_time = datetime.now() + timedelta(hours=duration_hours)
        scan_count = 0

        while datetime.now() < end_time:
            clear_output(wait=True)

            print(f"📊 DAILY CANDLESTICK SCANNER - Scan #{scan_count + 1}")
            print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"🎯 Monitoring until: {end_time.strftime('%H:%M:%S')}")
            print(f"📈 Timeframe: Daily candles for reliable patterns")
            print("=" * 80)

            if self.is_market_hours() or scan_count == 0:  # Always run first scan
                # Run scan
                self.run_scan()
                df = self.display_results()

                # Create visualization every 2nd scan
                if scan_count % 2 == 0 and df is not None:
                    self.create_visualization(df)

                scan_count += 1

                # Save results every 3 scans
                if scan_count % 3 == 0:
                    self.save_results(df)

                print(f"\n⏳ Next scan in 2 hours...")
                print(f"📊 Scans completed: {scan_count}")
                time.sleep(7200)  # Wait 2 hours between scans
            else:
                print("🌙 Market closed - waiting 1 hour...")
                time.sleep(3600)  # Wait 1 hour when market closed

# Initialize scanner
tickers = [ "NVDA","MSFT","AAPL","GOOGL","GOOG","AMZN","META","AVGO","TSLA","NFLX",
    "COST","PLTR","ASML","TMUS","CSCO","AMD","AZN","LIN","PEP","TXN",
    "BKNG","INTU","SHOP","PDD","QCOM","ISRG","AMGN","ADBE","APP","ARM",
    "GILD","HON","MU","AMAT","LRCX","CMCSA","ADI","PANW","ADP","MELI",
    "KLAC","SNPS","INTC","DASH","CRWD","VRTX","SBUX","MSTR","CEG","CDNS",
    "ORLY","CTAS","MDLZ","ABNB","MAR","PYPL","MRVL","CSX","ADSK","MNST",
    "AEP","AXON","NXPI","WDAY","FTNT","REGN","FAST","ROP","PCAR","IDXX",
    "PAYX","ROST","CPRT","EXC","DDOG","TEAM","BKR","EA","XEL","TTWO",
    "KDP","FANG","ZS","CHTR","CCEP","CSGP","VRSK","MCHP","CTSH","GEHC",
    "KHC","ODFL","WBD","DXCM","TTD","LULU","CDW","ON","BIIB","GFS"]

print("🔧 Setting up Colab Stock Scanner...")
scanner = ColabTechnicalScanner(tickers)

print("""
🚀 COLAB DAILY CANDLESTICK SCANNER READY!

📈 PROFESSIONAL DAILY PATTERN RECOGNITION!

USAGE OPTIONS:

1️⃣ SINGLE DAILY SCAN:
   results_df = scanner.run_scan()
   scanner.display_results()

2️⃣ START DAILY MONITORING (8 hours):
   scanner.start_monitoring(duration_hours=8)

3️⃣ FULL ANALYSIS WITH CHARTS:
   results_df = scanner.run_scan()
   df = scanner.display_results()
   scanner.create_visualization(df)

📊 Daily timeframe = More reliable candlestick patterns!
🎯 Original Setup A criteria: RSI 30-40, Volume 1.5x, 2% EMA distance
🔥 10+ Powerful reversal patterns with strength scoring!

Ready for professional daily analysis! 📈
""")

# Uncomment ONE of these to start:
# scanner.start_monitoring(duration_hours=4)  # Monitor for 4 hours with daily analysis
# results_df = scanner.run_scan(); df = scanner.display_results()  # Single daily scan

🔧 Setting up Colab Stock Scanner...

🚀 COLAB DAILY CANDLESTICK SCANNER READY!

📈 PROFESSIONAL DAILY PATTERN RECOGNITION!

USAGE OPTIONS:

1️⃣ SINGLE DAILY SCAN:
   results_df = scanner.run_scan()
   scanner.display_results()

2️⃣ START DAILY MONITORING (8 hours):
   scanner.start_monitoring(duration_hours=8)

3️⃣ FULL ANALYSIS WITH CHARTS:
   results_df = scanner.run_scan()
   df = scanner.display_results()
   scanner.create_visualization(df)

📊 Daily timeframe = More reliable candlestick patterns!
🎯 Original Setup A criteria: RSI 30-40, Volume 1.5x, 2% EMA distance
🔥 10+ Powerful reversal patterns with strength scoring!

Ready for professional daily analysis! 📈



In [13]:
results_list = scanner.run_scan() # run_scan returns a list of dictionaries
results_df = scanner.display_results() # display_results returns the DataFrame

🔍 Starting daily scan at 22:43:17
📊 Analyzing 100 tickers with daily candlestick patterns...
   Progress: 1/100
   Progress: 11/100
   Progress: 21/100
   Progress: 31/100
   Progress: 41/100
   Progress: 51/100
   Progress: 61/100
   Progress: 71/100
   Progress: 81/100
   Progress: 91/100
✅ Daily scan completed in 28.0 seconds
🎯 SETUP A QUALIFIED STOCKS
No stocks currently meet all Setup A criteria
🚀 HIGH MOMENTUM STOCKS (Top 15)

🔥 POWERFUL REVERSAL PATTERNS (2)
🔥 PDD: $128.21 | RSI: 80.97 | Pattern: Bullish Engulfing
🔥 NFLX: $1218.07 | RSI: 68.21 | Pattern: Bullish Engulfing

📊 PATTERN BREAKDOWN:
   Inverted Hammer: 5 stocks
   Bullish Engulfing: 3 stocks
   Hammer: 3 stocks
   Doji (Oversold): 1 stocks

👀 WATCH ZONE STOCKS (1)
⏰ CSGP: $90.51 | RSI: 32.98 | Waiting for signal

📈 SUMMARY
Total analyzed: 100
Setup A qualified: 0
Powerful patterns (3/3): 3
Strong patterns (2/3): 11
Trend confirmed: 36
High momentum (>50): 0
Average RSI: 52.9


In [10]:
# Filter the results_df DataFrame for stocks with the 'morning_star' pattern
morning_star_stocks = results_df[results_df['morning_star'] == True]

# Display the filtered stocks
print("Stocks with Morning Star pattern:")
if not morning_star_stocks.empty:
    display(morning_star_stocks[['ticker', 'price', 'rsi', 'momentum', 'patterns_found']])
else:
    print("No stocks found with Morning Star pattern in the latest scan.")

Stocks with Morning Star pattern:
No stocks found with Morning Star pattern in the latest scan.


Based on your notebook, "Setup A: Pullback Buy" on the **Daily** timeframe is defined by the following criteria:

1.  **Trend Confirmation**:
    *   50 EMA > 200 EMA
    *   Price is above the 21 EMA

2.  **Setup A: Pullback Buy**:
    *   Price touches or is near the 21 EMA (within 2% tolerance).
    *   RSI is between 30 and 40.
    *   A Bullish reversal candle is present (e.g., Hammer, Bullish Engulfing).
    *   Volume is 1.5 times the 20-day moving average volume.

The code also includes checks for additional bullish reversal patterns with a strength scoring system for the daily scan.

Based on the `calculate_daily_momentum` function in your notebook (in cell `wirtnDgTSYfe`), the momentum score is a calculated value that takes into account several factors to assess the strength and recent performance of a stock. It's a weighted combination of:

*   **RSI proximity to 35**: Stocks with RSI closer to 35 (ideal for a pullback buy) get a higher score in this component.
*   **Volume Ratio**: Higher volume relative to the 20-day average volume contributes positively to the score.
*   **5-day Price Change**: Recent price appreciation over the last 5 days is a significant factor.
*   **10-day Price Change**: Price change over the last 10 days also contributes, but with a lower weight than the 5-day change.
*   **EMA Relationship**: The stock's price position relative to the 21 EMA is also considered.

The score is then scaled to be between 0 and 100. A higher momentum score indicates stronger recent upward price movement and volume, particularly in the context of a potential pullback setup.

In [15]:
# Display the results DataFrame as a table
display(results_df)

,ticker,timestamp,price,ema_21,ema_50,ema_200,rsi,volume,volume_avg,trend_confirmed,...,ema21_distance_pct,bullish_engulfing,morning_star,three_white_soldiers,abandoned_baby,hammer,piercing_pattern,bullish_harami,inverted_hammer,dragonfly_doji
0,ASML,2025-08-25 22:43:17,754.46,738.91,740.46,731.68,74.82,729456,1453502,True,...,2.10,False,False,False,False,False,False,False,False,False
1,PCAR,2025-08-25 22:43:18,100.51,98.49,97.20,97.75,58.28,2265075,2386473,False,...,2.05,False,False,False,False,False,False,False,False,False
2,VRTX,2025-08-25 22:43:18,388.94,407.22,428.34,451.94,58.09,1094324,2495251,False,...,-4.49,False,False,False,False,False,False,False,False,False
3,KLAC,2025-08-25 22:43:18,879.55,895.24,880.86,795.18,49.51,462559,1061467,False,...,-1.75,True,False,False,False,False,False,False,False,False
4,GEHC,2025-08-25 22:43:18,74.59,73.73,73.64,76.22,65.24,1988451,3980662,False,...,1.16,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,PEP,2025-08-25 22:43:44,148.20,146.19,141.90,142.38,70.04,6620332,7657746,False,...,1.37,False,False,False,False,False,False,False,False,False
96,ODFL,2025-08-25 22:43:44,155.47,153.21,156.56,167.69,59.12,1692013,2247175,False,...,1.48,False,False,False,False,False,False,False,False,False
97,TTWO,2025-08-25 22:43:44,231.83,229.40,229.71,216.67,56.79,1173141,1862542,True,...,1.06,False,False,False,False,False,False,False,False,False
98,WBD,2025-08-25 22:43:44,12.04,11.96,11.67,10.57,42.48,31905113,49051055,True,...,0.68,False,False,False,False,False,False,False,False,False
